# Day 5: Fluid property tables, files, and your own interpolation

**Time:** about 1 hour |  **Needs:** Days 1 to 4
**Reservoir idea:** PVT (pressure-volume-temperature) tables: how the oil's formation volume factor $B_o$ and viscosity $\mu_o$ change with pressure
**Python you will learn:** writing and reading files, lists in depth (methods, copies), lists of lists, reading a table into Python, writing your own `interp()` function
**This is also the review day of Week 1**: the last section pulls Days 1 to 5 together.

By the end of today you will be able to:

1. write text to a file and read it back, and cope with a missing file or a bad line,
2. use list methods and avoid the classic trap of two names for the same list,
3. store a table as a list of rows and pick out a column,
4. write the linear-interpolation function that every simulator uses to read tables,
5. read a production log with some messy lines and summarise it.

> **Companion reading (optional, about 50 minutes).** Charles Severance, *Python for Everybody* (free online: <https://www.py4e.com/html3/>):
> - Chapter 7 "Files": "Persistence", "Opening files", "Text files and lines", "Reading files", "Searching through a file", "Letting the user choose the file name", "Using try, except, and open", "Writing files"
> - Chapter 8 "Lists": "A list is a sequence", "Lists are mutable", "Traversing a list", "List operations", "List slices", "List methods", "Lists and strings", "Parsing lines", "Objects and values", "Aliasing", "List arguments"

## 0. Setup and recap

Run the next cell. It also defines two helper functions from Day 4 (given here, so that today stands alone).

**Recap of Day 4.** Strings are sequences (indexes, slices, methods). `.split()` cuts a line into words; `float(word)` turns a word into a number; f-strings format output; constants are written in capitals.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

BBL_TO_FT3 = 5.615        # ft3 per barrel


def ooip_stb(bulk_volume, phi, sw, bo, ft3_per_bbl=BBL_TO_FT3):
    """Original oil in place (STB) from bulk volume (ft3), porosity, water saturation and Bo (rb/STB)."""
    return bulk_volume * phi * (1 - sw) / (ft3_per_bbl * bo)

print("ready")

## 1. Story: the table that lived in an email

*(An invented but typical situation.)* A PVT laboratory sends its report as a PDF. Somebody copies the table into an email, somebody else pastes it into a spreadsheet, and a third person types the numbers into the simulator input by hand. Three weeks later, two engineers have different values for $B_o$ at 3500 psi, and nobody knows which copy is right.

Good practice is the opposite: **keep the data in one plain text file, and let programs read it**. Then there is one source of truth, and a program can check the file for mistakes. Today you learn to do exactly that.

## 2. What is a PVT table?

Oil in the reservoir is under pressure and has gas dissolved in it. Two properties change with pressure, and the simulator needs both:

| Property | Symbol | Unit | Meaning |
|---|---|---|---|
| oil formation volume factor | $B_o$ | rb/STB | reservoir barrels of oil that give 1 stock-tank barrel at the surface (always above 1: the oil shrinks) |
| oil viscosity | $\mu_o$ | cp | how hard the oil is to push through the rock |

(Water has $B_w$ and $\mu_w$ too.) The laboratory measures them at a handful of pressures, and a **table** lists the values. Here is the table used in the thesis (Application 1): pressure in psi, then $B_o$, $B_w$, $\mu_o$ and $\mu_w$.

| P | BO | BW | MUO | MUW |
|---|---|---|---|---|
| 1500 | 1.522 | 1.0044 | 0.45 | 0.95 |
| 2000 | 1.5 | 1.0 | 0.5 | 1.0 |
| 3500 | 1.478 | 0.9956 | 0.55 | 1.05 |
| 4300 | 1.45 | 0.9911 | 0.75 | 1.07 |

The simulator needs $B_o$ at *any* pressure, such as 2750 psi, which is not in the table. We will estimate it by drawing a straight line between the two nearest table points. That is **linear interpolation**, and it is the aim of today.

## 3. Files: writing and reading

A **file** keeps data after the program ends. We will work with plain text files. Three steps: *open* the file, *use* it, *close* it. The `with` statement does the closing for you, even if something goes wrong in between:

```python
with open("name.txt", "w") as f:      # "w" = write (creates or overwrites the file)
    f.write("first line\n")           # \n marks the end of a line
    f.write("second line\n")
```

| Mode | Meaning |
|---|---|
| `"r"` | read (default); the file must exist |
| `"w"` | write; **erases** an existing file |
| `"a"` | append: add to the end of an existing file |

First we create the PVT file, with some comment lines and a blank line, as real files have. (In JupyterLab, the **file browser** on the left shows the files appear.)

In [ ]:
pvt_text = '''# PVT table from the lab report (thesis Application 1)
# P in psi, Bo and Bw in rb/STB, viscosities in cp

P,BO,BW,MUO,MUW
1500,1.522,1.0044,0.45,0.95
2000,1.5,1.0,0.5,1.0
3500,1.478,0.9956,0.55,1.05
4300,1.45,0.9911,0.75,1.07
'''

with open("pvt.csv", "w") as f:
    f.write(pvt_text)

print("file written")

Now read it back. A file is a sequence of **lines**, and `for line in f:` gives one line at a time, *including the end-of-line character*, so we usually apply `.strip()`.

In [ ]:
with open("pvt.csv") as f:
    for line in f:
        print(line.strip())

Other ways to read: `f.read()` returns the whole file as one string; `f.readlines()` returns a list of the lines.

What if the file does not exist? Python raises a `FileNotFoundError`. You can catch it with `try` / `except`, as on Day 2:

In [ ]:
try:
    with open("no_such_file.csv") as f:
        text = f.read()
except FileNotFoundError:
    print("That file does not exist: check the name and the folder")

**Your turn 3.1.** The list `well_names = ["P1", "P2", "I1"]` holds the wells of a field. Write them to a file `wells.txt`, **one name per line**.

In [ ]:
well_names = ["P1", "P2", "I1"]

# YOUR CODE HERE

In [ ]:
# Check (do not edit)
with open("wells.txt") as f:
    lines_back = f.read().splitlines()
assert lines_back == well_names, f"wells.txt should hold one name per line; reading it back gave {lines_back}. Did you add '\\n' after each name?"
print("Correct")

<details><summary>Solution</summary>

```python
with open("wells.txt", "w") as f:
    for name in well_names:
        f.write(name + "\n")
```
</details>

**Your turn 3.2.** Open `pvt.csv` and count its lines: `n_lines` (all lines), `n_comments` (lines that start with `#`) and `n_blank` (lines with nothing but spaces, after `.strip()`).

In [ ]:
n_lines = 0
n_comments = 0
n_blank = 0

# YOUR CODE HERE

In [ ]:
# Check (do not edit)
assert n_lines == 8, f"n_lines should be 8, got {n_lines}"
assert n_comments == 2, f"n_comments should be 2, got {n_comments}"
assert n_blank == 1, f"n_blank should be 1, got {n_blank}"
print("Correct")

<details><summary>Solution</summary>

```python
with open("pvt.csv") as f:
    for line in f:
        line = line.strip()
        n_lines += 1
        if line.startswith("#"):
            n_comments += 1
        elif line == "":
            n_blank += 1
```
</details>

## 4. Lists in depth

You have used lists since Day 1. Here is the rest of what you need. Predict the output of each cell before you run it.

### 4.1 Operations and methods

| You want to... | Write | Note |
|---|---|---|
| join two lists | `a + b` | makes a new list |
| repeat | `[0] * 5` | `[0, 0, 0, 0, 0]` |
| is an item in it? | `3 in a` | `True` / `False` |
| add one item at the end | `a.append(x)` | changes `a` |
| add several | `a.extend([x, y])` | changes `a` |
| remove and return the last / item `i` | `a.pop()` / `a.pop(i)` | changes `a` |
| remove the first item equal to `x` | `a.remove(x)` | changes `a` |
| sort in place | `a.sort()` | changes `a`; `sorted(a)` returns a new list |
| position of an item | `a.index(x)` | `ValueError` if missing |
| text to list and back | `"a b".split()` and `" ".join(["a", "b"])` | |

In [ ]:
pressures = [3000, 2500, 2800]
pressures.append(2200)
pressures.sort()
print(pressures)
print(pressures.index(2800))
print(pressures.pop())
print(pressures)
print(" - ".join(["P1", "P2", "I1"]))

### 4.2 A trap: two names for one list

Predict what `a` holds at the end of this cell. (Is it `[1, 2, 3]`, or `[1, 2, 3, 99]`?)

In [ ]:
a = [1, 2, 3]
b = a
b.append(99)
print("a =", a)
print("b =", b)

`b = a` does **not** copy the list. It gives the *same* list a second name. Changing it through `b` changes it for `a`, too. To get a real copy, write `b = a.copy()` (or `list(a)`, or `a[:]`):

In [ ]:
a = [1, 2, 3]
b = a.copy()
b.append(99)
print("a =", a)
print("b =", b)

This matters when you pass a list **to a function**: the function receives the list itself, not a copy. If the function changes it (`.append`, `.sort`, ...), the caller's list changes. Functions that should leave their input alone must build a **new** list.

**Your turn 4.1.** Write `column(rows, j)` that returns a new list with item number `j` of every row. For `rows = [[1500, 1.522], [2000, 1.5], [3500, 1.478]]`, `column(rows, 1)` is `[1.522, 1.5, 1.478]`. (Rows are lists, so `row[j]` works.)

In [ ]:
def column(rows, j):
    """New list with item j of each row."""
    # YOUR CODE HERE
    pass

In [ ]:
# Check (do not edit)
rows = [[1500, 1.522], [2000, 1.5], [3500, 1.478]]
assert column(rows, 0) == [1500, 2000, 3500], f"column(rows, 0) gave {column(rows, 0)}"
assert column(rows, 1) == [1.522, 1.5, 1.478], f"column(rows, 1) gave {column(rows, 1)}"
assert rows == [[1500, 1.522], [2000, 1.5], [3500, 1.478]], "the function must not change the rows"
print("Correct")

<details><summary>Solution</summary>

```python
def column(rows, j):
    """New list with item j of each row."""
    result = []
    for row in rows:
        result.append(row[j])
    return result
```
</details>

## 5. A table as a list of lists

A table is a list of **rows**, and each row is itself a list. Item `table[2][1]` means: row 2, then item 1 of that row.

In [ ]:
table = [[1500, 1.522], [2000, 1.5], [3500, 1.478], [4300, 1.45]]

print(table[0])
print(table[2][1])          # Bo at 3500 psi
print(len(table), "rows")
for row in table:
    print(f"P = {row[0]:>5}   Bo = {row[1]}")

Now make the computer read `pvt.csv` into this shape. The steps for each line: strip it, skip it if it is blank or a comment, cut it at the commas (`.split(",")`). The **first** surviving line is the header (names, kept as text); all the others are numbers, converted with `float`.

**Your turn 5.1.** Write `read_table(filename)` that returns two things: `header` (a list of column names) and `rows` (a list of rows, each a list of floats).

In [ ]:
def read_table(filename):
    """Read a comma-separated table. Returns (header, rows); blank lines and lines starting with # are skipped."""
    header = None
    rows = []
    # YOUR CODE HERE
    return header, rows

In [ ]:
# Check (do not edit)
header, rows = read_table("pvt.csv")
assert header == ["P", "BO", "BW", "MUO", "MUW"], f"header should be ['P', 'BO', 'BW', 'MUO', 'MUW'], got {header}"
assert len(rows) == 4, f"there should be 4 data rows, got {len(rows)}"
assert rows[0] == [1500.0, 1.522, 1.0044, 0.45, 0.95], f"first row is wrong: {rows[0]}"
assert rows[-1][0] == 4300.0, "the last row should be the 4300 psi row"
print("Correct")

<details><summary>Solution</summary>

```python
def read_table(filename):
    """Read a comma-separated table. Returns (header, rows); blank lines and lines starting with # are skipped."""
    header = None
    rows = []
    with open(filename) as f:
        for line in f:
            line = line.strip()
            if line == "" or line.startswith("#"):
                continue
            words = line.split(",")
            if header is None:
                header = words
            else:
                row = []
                for word in words:
                    row.append(float(word))
                rows.append(row)
    return header, rows
```
</details>

With the table in memory, columns are easy: `column(rows, 0)` is the list of pressures and `column(rows, 1)` the list of $B_o$ values. (Later, the library `pandas` will do this reading in one line, and the standard `csv` module also exists; writing it yourself once shows you what they do.)

In [ ]:
header, rows = read_table("pvt.csv")
p_table = column(rows, 0)
bo_table = column(rows, 1)
muo_table = column(rows, 3)

print(header)
print("pressures:", p_table)
print("Bo       :", bo_table)

## 6. Linear interpolation

### The idea

To find $B_o$ at 2750 psi, look at the two table points around it: 2000 psi ($B_o = 1.5$) and 3500 psi ($B_o = 1.478$). 2750 is exactly halfway (750 of the 1500 psi between them), so we take the value halfway between 1.5 and 1.478:

$$B_o(2750) = 1.5 + 0.5\times(1.478-1.5) = 1.489$$

In general, if $x$ lies between table points $x_i$ and $x_{i+1}$:

$$y = y_i + \frac{x-x_i}{x_{i+1}-x_i}\,\big(y_{i+1}-y_i\big) \tag{1}$$

The fraction $(x-x_i)/(x_{i+1}-x_i)$ is 0 at $x_i$ and 1 at $x_{i+1}$.

### Outside the table

What about 1000 psi, below the first table point, or 5000 psi, above the last? The thesis (Chapter 5, section 15) uses the simplest rule: **flat extrapolation**. Below the first point return the first value; above the last point return the last value. So the three cases are exactly the `if` / `elif` / `else` of Day 2's bridge:

1. `x` at or below the first table value: return the first `y`,
2. `x` at or above the last: return the last `y`,
3. otherwise find the pair of neighbours around `x` and use equation (1).

**Your turn 6.1.** Write `interp(x, xs, ys)`: `xs` is the list of table positions (increasing) and `ys` the list of values.

In [ ]:
def interp(x, xs, ys):
    """Linear interpolation of the table (xs, ys) at x, with flat extrapolation outside the table."""
    # YOUR CODE HERE
    pass

In [ ]:
# Check (do not edit)
assert interp(2750, p_table, bo_table) == 1.489 or abs(interp(2750, p_table, bo_table) - 1.489) < 1e-12, f"Bo at 2750 psi should be 1.489, got {interp(2750, p_table, bo_table)}"
assert interp(2000, p_table, bo_table) == 1.5, "at a table point the table value is returned"
assert interp(1000, p_table, bo_table) == 1.522, "below the table: first value (flat extrapolation)"
assert interp(5000, p_table, bo_table) == 1.45, "above the table: last value (flat extrapolation)"
for p in range(1400, 4500, 137):                      # compare with NumPy's own interpolation
    assert abs(interp(p, p_table, bo_table) - np.interp(p, p_table, bo_table)) < 1e-12, f"disagrees with np.interp at {p} psi"
print("Correct: your interp agrees with np.interp everywhere we tried")

<details><summary>Solution</summary>

```python
def interp(x, xs, ys):
    """Linear interpolation of the table (xs, ys) at x, with flat extrapolation outside the table."""
    if x <= xs[0]:
        return ys[0]
    if x >= xs[-1]:
        return ys[-1]
    for i in range(len(xs) - 1):
        if xs[i] <= x <= xs[i + 1]:
            fraction = (x - xs[i]) / (xs[i + 1] - xs[i])
            return ys[i] + fraction * (ys[i + 1] - ys[i])
```
(A function can have several `return` lines; the first one reached ends it.)
</details>

Now use it. **Your turn 6.2.** Store in `bo_2750` and `muo_2750` the oil formation volume factor and viscosity at 2750 psi, and in `bo_1000` the $B_o$ at 1000 psi (below the table).

In [ ]:
# YOUR CODE HERE
bo_2750 = None
muo_2750 = None
bo_1000 = None

In [ ]:
# Check (do not edit)
assert abs(bo_2750 - 1.489) < 1e-9, f"bo_2750 should be 1.489, got {bo_2750}"
assert abs(muo_2750 - 0.525) < 1e-9, f"muo_2750 should be 0.525, got {muo_2750}"
assert bo_1000 == 1.522, f"bo_1000 should be 1.522 (flat extrapolation), got {bo_1000}"
print("Correct")

<details><summary>Solution</summary>

```python
bo_2750 = interp(2750, p_table, bo_table)
muo_2750 = interp(2750, p_table, muo_table)
bo_1000 = interp(1000, p_table, bo_table)
```
</details>

See the result: the table points as dots, and your function's values on a fine pressure grid as a line. Notice the flat parts outside the table.

In [ ]:
p_fine = []
bo_fine = []
mu_fine = []
for p in range(1000, 5001, 50):
    p_fine.append(p)
    bo_fine.append(interp(p, p_table, bo_table))
    mu_fine.append(interp(p, p_table, muo_table))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(p_fine, bo_fine, label="interp()")
ax1.plot(p_table, bo_table, "o", label="table")
ax1.set_xlabel("pressure (psi)"); ax1.set_ylabel("Bo (rb/STB)"); ax1.legend(); ax1.grid(True)
ax2.plot(p_fine, mu_fine, label="interp()")
ax2.plot(p_table, muo_table, "o", label="table")
ax2.set_xlabel("pressure (psi)"); ax2.set_ylabel("oil viscosity (cp)"); ax2.legend(); ax2.grid(True)
plt.tight_layout()
plt.show()

## 7. A messy production log

Real data is untidy. Below we create a small production log of two wells over 30 days, with **two bad lines** (a missing value and a text value), like a log exported from a field system. The cell uses a loop, `f"..."` strings and `f.write`, all things you know. You do not need to study how the bad lines are put in; just run it.

In [ ]:
lines = ["# date well oil_rate(STB/d) water_rate(STB/d)"]
for d in range(30):
    date = f"2024-01-{d + 1:02d}"
    p1_oil = round(2500 * 0.985 ** d)
    p2_oil = round(1800 * 0.99 ** d)
    p1_water = 10 * d
    p2_water = 3 * d

    if d == 7:
        lines.append(f"{date} P1 n/a {p1_water}")            # a text value
    else:
        lines.append(f"{date} P1 {p1_oil} {p1_water}")

    if d == 15:
        lines.append(f"{date} P2 {p2_oil}")                   # water rate missing
    else:
        lines.append(f"{date} P2 {p2_oil} {p2_water}")

with open("production_log.txt", "w") as f:
    f.write("\n".join(lines) + "\n")

print(len(lines), "lines written. First lines:")
for line in lines[:4]:
    print(line)

**Your turn 7.1.** Write `read_log(filename)` that returns two things: `records`, a list with one entry `[date, well, oil, water]` for each good line (date and well as text, oil and water as floats), and `n_bad`, the number of lines that could not be read. Skip comment and blank lines (they are *not* bad). A line is bad if `float()` fails (`ValueError`) or a field is missing (`IndexError`, when you ask for `words[3]` on a line with only three words).

(You can catch two kinds of error at once: `except (ValueError, IndexError):`.)

In [ ]:
def read_log(filename):
    """Read the production log. Returns (records, n_bad)."""
    records = []
    n_bad = 0
    # YOUR CODE HERE
    return records, n_bad

In [ ]:
# Check (do not edit)
records, n_bad = read_log("production_log.txt")
assert n_bad == 2, f"n_bad should be 2 (one text value, one missing field), got {n_bad}"
assert len(records) == 58, f"there should be 58 good records (60 data lines minus 2 bad), got {len(records)}"
assert records[0] == ["2024-01-01", "P1", 2500.0, 0.0], f"first record is wrong: {records[0]}"
print("Correct")

<details><summary>Solution</summary>

```python
def read_log(filename):
    """Read the production log. Returns (records, n_bad)."""
    records = []
    n_bad = 0
    with open(filename) as f:
        for line in f:
            line = line.strip()
            if line == "" or line.startswith("#"):
                continue
            words = line.split()
            try:
                oil = float(words[2])
                water = float(words[3])
            except (ValueError, IndexError):
                n_bad += 1
                continue
            records.append([words[0], words[1], oil, water])
    return records, n_bad
```
</details>

**Your turn 7.2.** Using `records`, find with the loop patterns of Day 3: `total_p1` and `total_p2` (the total oil produced by each well, in STB; the rates are STB/day and there is one record per day, so the total is the sum of the rates) and `max_water_p2` (the highest water rate of well `P2`). The well name is `record[1]`, the oil rate `record[2]` and the water rate `record[3]`.

In [ ]:
total_p1 = 0
total_p2 = 0
max_water_p2 = 0

# YOUR CODE HERE

In [ ]:
# Check (do not edit)
exp_p1 = 0
exp_p2 = 0
for d in range(30):
    if d != 7:
        exp_p1 += round(2500 * 0.985 ** d)
    if d != 15:
        exp_p2 += round(1800 * 0.99 ** d)
assert total_p1 == exp_p1, f"total_p1 should be {exp_p1}, got {total_p1}"
assert total_p2 == exp_p2, f"total_p2 should be {exp_p2}, got {total_p2}"
assert max_water_p2 == 3 * 29, f"max_water_p2 should be 87 (3 x 29), got {max_water_p2}"
print(f"Correct: P1 produced {total_p1:,.0f} STB, P2 {total_p2:,.0f} STB")

<details><summary>Solution</summary>

```python
for record in records:
    if record[1] == "P1":
        total_p1 += record[2]
    elif record[1] == "P2":
        total_p2 += record[2]
        if record[3] > max_water_p2:
            max_water_p2 = record[3]
```
</details>

You just read a real-looking file, survived bad lines, and summarised it. Notice what is clumsy: we had to test the well name with `if`/`elif` for every well. With 200 wells you would want a *dictionary* (a lookup from well name to total). That is Day 6.

## 8. Week 1 review

This section brings the week together. Here is what Week 1 gave you:

| Day | You can now... |
|---|---|
| 1 | explain a model, store values in variables, use lists and a `for` loop, make a plot |
| 2 | decide with `if`/`elif`/`else`, write functions, catch errors with `try`/`except` |
| 3 | repeat with `while`, count/sum/find maximum, see numerical instability |
| 4 | compute volumes and OOIP in the right units, work with strings and parse text |
| 5 | read and write files, handle lists of lists, interpolate a table, read a messy log |

**Your turn 8.1 (a small project).** The thesis reservoir starts at a datum pressure of 4800 psi. Write `ooip_at_pressure(pressure_psi)` that

1. reads `pvt.csv` with your `read_table`,
2. takes the `BO` column,
3. interpolates $B_o$ at the given pressure with your `interp`,
4. returns the OOIP of the thesis reservoir (bulk volume 14.4 million ft³, $\phi = 0.2$, $S_w = 0.2$) with the `ooip_stb` function above.

Then store `ooip_4800` and `ooip_2750` using it.

Think before you code: 4800 psi is **above** the last table point (4300 psi). What $B_o$ will your `interp` use? What does that say about trusting results outside a table?

In [ ]:
def ooip_at_pressure(pressure_psi):
    """OOIP (STB) of the thesis reservoir if Bo is taken from pvt.csv at the given pressure."""
    # YOUR CODE HERE
    pass

ooip_4800 = None
ooip_2750 = None

In [ ]:
# Check (do not edit)
ooip_4800 = ooip_at_pressure(4800)
ooip_2750 = ooip_at_pressure(2750)
expected_4800 = 14.4e6 * 0.2 * 0.8 / (5.615 * 1.45)
expected_2750 = 14.4e6 * 0.2 * 0.8 / (5.615 * 1.489)
assert abs(ooip_4800 - expected_4800) < 1, f"ooip_4800 should be about {expected_4800:,.0f}, got {ooip_4800:,.0f}"
assert abs(ooip_2750 - expected_2750) < 1, f"ooip_2750 should be about {expected_2750:,.0f}, got {ooip_2750:,.0f}"
print(f"Correct: OOIP at 4800 psi = {ooip_4800:,.0f} STB, at 2750 psi = {ooip_2750:,.0f} STB")

<details><summary>Solution</summary>

```python
def ooip_at_pressure(pressure_psi):
    """OOIP (STB) of the thesis reservoir if Bo is taken from pvt.csv at the given pressure."""
    header, rows = read_table("pvt.csv")
    bo = interp(pressure_psi, column(rows, 0), column(rows, 1))
    return ooip_stb(bulk_volume=14.4e6, phi=0.2, sw=0.2, bo=bo)
```
Above 4300 psi your `interp` returns the last table value (1.45), so the answer at 4800 psi is the answer at 4300 psi. The table says nothing about higher pressures. In real work, extrapolating far beyond a lab table is a risk, and good practice is to flag it with a warning.

(Strictly, OOIP does not depend on the current pressure in this way. It is calculated at initial conditions. The function shows how a table lookup plugs into a volume calculation, and how sensitive the result is to $B_o$: 1.45 against 1.5 changes the answer by 3 %.)
</details>

### Week 1 self-assessment

Tick the ones you can do *without looking*:

- [ ] write a `for` loop and a `while` loop and say when to use which
- [ ] get the first, last and a slice of a list or a string
- [ ] write a function with parameters and a `return`
- [ ] explain `print` vs `return`
- [ ] use `if`/`elif`/`else` and `try`/`except`
- [ ] open a file, loop over its lines, and split a line into numbers
- [ ] explain why `b = a` does not copy a list
- [ ] explain what linear interpolation does and where flat extrapolation applies

If more than three are unticked, repeat the exercises of that day before starting Week 2. The first time through, struggling is normal.

## 9. Bridge: tables inside `kusim`

Open `src/kusim/tables.py`. Its class `Table` stores a table and looks values up:

```python
def lookup(self, x):
    return np.interp(x, self.x, self.y)
```

`np.interp` is the NumPy function you compared with in Section 6: the same linear interpolation, with the same flat extrapolation. The docstring of the file lists the three cases of the thesis exactly as in your `interp`. Because your function agrees with it everywhere we checked, you could swap it in.

Two details are worth knowing:

1. The simulator needs not only the value but also the **slope** of the table (for example how $1/B_o$ changes with pressure, which appears in the storage coefficients). `Table.slope` returns the slope of the table segment that contains $x$, i.e. $(y_{i+1}-y_i)/(x_{i+1}-x_i)$, the fraction in your equation (1).
2. `kusim` stores $1/B$ (not $B$), because the equations use $1/B$. Interpolating $1/B$ is not exactly the same as interpolating $B$ and then inverting, but the difference is small.

And the thesis case itself uses today's observation: the datum pressure in `legacy/fortran-3d3/INPUT.DAT` is 4800 psi while the last PVT row is at 4300 psi, so the first calculations use the flat extrapolation rule you wrote.

The file reading in `src/kusim/io.py` does what `read_table` does: open, loop over the lines, split into words, convert to numbers. It reads the original `INPUT.DAT` format, where numbers are read by position and labels after them are ignored.

## 10. Self-quiz

Write short answers as comments, then open the answers.

1. What is the difference between mode `"w"` and mode `"a"` in `open`?
2. Why do we use `with open(...) as f:`?
3. `a = [3, 1, 2]`, `b = a`, `b.sort()`. What is `a`? How would you sort without changing `a`?
4. `table = [[1, 10], [2, 20], [3, 30]]`. What are `table[1]`, `table[2][0]` and `len(table[0])`?
5. In interpolation, what is returned for $x$ above the last table value, and why is that risky?

In [ ]:
# 1.
# 2.
# 3.
# 4.
# 5.

<details><summary>Answers</summary>

1. `"w"` erases the file (if it exists) and starts empty; `"a"` keeps the contents and adds to the end.
2. It closes the file automatically when the block ends, even if an error occurs, so data is not lost and the file is not left locked.
3. `a` is `[1, 2, 3]` too: `b` and `a` are the same list. Use `sorted(a)` (returns a new sorted list) or `b = a.copy()` first.
4. `[2, 20]`, `3` and `2`.
5. The last table value (flat extrapolation). Beyond the data the real property may keep changing (e.g. oil viscosity keeps rising with pressure); the result is a guess, not a measurement.
</details>

## 11. Pocket reference (today's Python)

| You want to... | Write |
|---|---|
| write a file | `with open("f.txt", "w") as f:` then `f.write("text\n")` |
| add to a file | `open("f.txt", "a")` |
| read lines | `with open("f.txt") as f:` then `for line in f:` |
| the whole file / list of lines | `f.read()` / `f.readlines()` |
| a missing file | `try: ... except FileNotFoundError:` |
| tidy a line | `line.strip()`, `line.startswith("#")`, `line.split(",")` |
| copy a list | `b = a.copy()` (not `b = a`) |
| list methods | `.append`, `.extend`, `.pop`, `.remove`, `.sort`, `.index` |
| join text | `" ".join(list_of_words)` |
| item of a table | `table[row][column]` |
| two kinds of error at once | `except (ValueError, IndexError):` |
| compare with a library | `np.interp(x, xs, ys)` |

## 12. Wrap-up

Today you:

- wrote and read text files, and handled a missing file and bad lines,
- learned list methods, and the trap of two names for one list,
- stored a table as a list of rows and read a column,
- wrote `interp()`, the table lookup at the heart of the simulator, and checked it against NumPy,
- read a messy production log and summarised it,
- reviewed the whole of Week 1.

**Next, Week 2, Day 6:** dictionaries and tuples. A dictionary will replace the clumsy `if`/`elif` of Section 7 (a total per well, however many wells there are), and you will start on rock properties: relative permeability curves.